# M3.9 · Eğitim pratiği — kapasite ve aşırı uyum, model seçimi, transfer öğrenme, veri sızıntısı

Derin ağ eğitmenin mühendislik tarafı. Daha geniş ve daha derin tam bağlı ağlar (784-300-100-10'dan 7 katmana: ders alıştırması 4) eğitim kaybını sıfıra indirirken doğrulama kaybının yükselmesi — aşırı uyum; erken durdurma ve kontrol noktası; k-katlı çapraz doğrulama ve öğrenme eğrileri. Önceden eğitilmiş katmanları yeniden kullanmak: alt katmanları dondurma, yalnız başlığı eğitme, kademeli çözme ve ince ayar; benzer görevlerde transferin neden işe yaradığı. Veri sızıntısı: ölçekleme istatistiğini testten öğrenmek, tekrarlanan/gruplu örneklerin iki tarafa düşmesi, zaman serisinde rastgele bölme. Saf Python'da 1-NN ile grup sızıntısı, NumPy'da k-katlı CV ve erken durdurma, PyTorch'ta kapasite deneyi ve transfer, sonda tüm katlar × kapasiteleri tek seferde eğiten toplu CV.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m3-9-training-practice.html](https://m-gonul.github.io/pytorch-lab/m3-9-training-practice.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m3_9_training_practice.py
CHECKS = {
 "b": "from labcheck import no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def t():\n        X, y, g = need(ns, \"make_groups\")(30, 4, 1)\n        ensure(len(X) == 120 and len(set(g)) == 30, \"boyutlar\")\n        tr, te = need(ns, \"split_groups\")(g, 0.7, 0)\n        ensure(not ({g[i] for i in tr} & {g[i] for i in te}), \"grup bölmesinde ortak grup olmamalı\")\n        tr, te = need(ns, \"split_random\")(120, 0.7, 0)\n        ensure(sorted(tr + te) == list(range(120)), \"rastgele bölme tam\")\n        ensure(need(ns, \"nn1_predict\")([[0, 0], [5, 5]], [0, 1], [[1, 1], [4, 6]]) == [0, 1], \"1-NN\")\n        rows = []\n        accs = []\n        for s in range(5):\n            X, y, g = ns[\"make_groups\"](60, 5, s)\n            a, b = need(ns, \"accuracy_both\")(X, y, g, s)\n            accs.append((a, b))\n            rows.append([s, f\"{a:.2f}\", f\"{b:.2f}\"])\n        ra = sum(a for a, _ in accs) / 5\n        gb = sum(b for _, b in accs) / 5\n        ensure(ra > 0.9 and gb < 0.8, \"rastgele bölme şişirilmiş, grup bölmesi gerçekçi\")\n        R_.table(\"1-NN, 60 denek × 5 tekrar ölçüm\", [\"tohum\", \"rastgele bölme\", \"grup (denek) bölmesi\"], rows,\n                 note=\"Rastgele bölmede test örneğinin 'ikizi' eğitimde: 1-NN onu ezberden bilir. Gerçek soru 'yeni bir denekte ne olur?' ise bölme denek düzeyinde yapılmalı (GroupKFold).\")\n    R_.test(\"grup sızıntısı\", t)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngü kuralları\", lambda: (max_loops(src, 0, func=\"early_stop\"), max_loops(src, 0, func=\"standardize_ok\")))\n\n    def t():\n        f = need(ns, \"kfold_indices\")(103, 5, 0)\n        ensure(len(f) == 5 and sorted(np.concatenate([v for _, v in f]).tolist()) == list(range(103)), \"katlar ayrık ve tam\")\n        ensure(all(len(np.intersect1d(a, b)) == 0 for a, b in f), \"eğitim/doğrulama ayrık\")\n        ensure(need(ns, \"early_stop\")([1.0, 0.8, 0.7, 0.72, 0.75, 0.71, 0.9], 2) == (4, 2), \"erken durdurma (4. epoch'ta dur, 2. epoch en iyi)\")\n        ensure(ns[\"early_stop\"]([3.0, 2, 1, 0.5], 2) == (3, 3), \"hiç durmazsa son epoch\")\n        rng = np.random.default_rng(0)\n        x = rng.uniform(-1, 1, 60)\n        y = np.sin(3 * x) + 0.2 * rng.normal(size=60)\n        cv = need(ns, \"poly_cv\")(x, y, [1, 3, 5, 9, 15])\n        best = min(cv, key=cv.get)\n        ensure(3 <= best <= 9 and cv[15] > cv[best], \"orta derece seçilmeli\")\n        Xtr = rng.normal(0, 1, (50, 3))\n        Xte = rng.normal(3, 1, (50, 3))\n        a, _ = need(ns, \"standardize_leak\")(Xtr, Xte)\n        b, _ = need(ns, \"standardize_ok\")(Xtr, Xte)\n        close(b.mean(0), np.zeros(3), atol=1e-9, what=\"eğitimde ortalama 0\")\n        ensure(abs(a.mean()) > 0.5, \"sızıntılı ölçek eğitim dağılımını kaydırır\")\n        R_.table(\"Polinom derecesi, 5 katlı CV (n = 60)\", [\"derece\", \"CV MSE\"], [[d, f\"{v:.3f}\"] for d, v in cv.items()],\n                 note=\"CV hatası klasik U çizer. Seçilen dereceyle tüm eğitim verisinde yeniden eğit, testte bir kez değerlendir.\")\n    R_.test(\"CV, erken durdurma, ölçekleme\", t)\n",
 "d": "import torch\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def cap():\n        runs = need(ns, \"capacity_runs\")(40, 0)\n        ensure(len(runs) == 3 and all(len(v) == 40 for v in runs.values()), \"üç model, 40 epoch\")\n        rows = []\n        for name, h in runs.items():\n            best = min(range(40), key=lambda i: h[i][1])\n            rows.append([name, f\"{h[-1][0]:.4f}\", f\"{h[-1][1]:.3f}\", f\"{h[best][1]:.3f} ({best + 1})\", f\"{h[-1][2]:.3f}\"])\n            ensure(h[-1][1] > h[best][1] * 1.05, f\"{name}: doğrulama kaybı en iyiden sonra yükselmeli\")\n        big = list(runs.values())[-1]\n        small = list(runs.values())[0]\n        ensure(big[-1][1] > small[-1][1], \"en büyük model en çok aşırı uyar\")\n        R_.table(\"Ders alıştırması 4 (digits ile): 40 epoch\", [\"mimari\", \"son eğitim kaybı\", \"son doğrulama kaybı\", \"en iyi doğrulama (epoch)\", \"son doğrulama doğruluğu\"], rows,\n                 note=\"Eğitim ve doğrulama eğrilerinin ayrışması = aşırı uyum. Büyük model en iyi noktasına daha erken ulaşır, sonra daha hızlı bozulur; doğruluk benzer kalırken kayıp kötüleşir (aşırı güvenli yanlışlar). Çare: erken durdurma, düzenleme (M3.7), daha çok veri.\")\n    R_.test(\"kapasite ve aşırı uyum\", cap)\n\n    def tr():\n        keys = (\"sıfırdan\", \"dondurulmuş\", \"ince ayar\", \"ilgisiz kaynak\")\n        acc = {k: 0.0 for k in keys}\n        for sd in range(3):\n            r = need(ns, \"transfer\")(100, sd)\n            for k in keys:\n                acc[k] += r[k] / 3\n        ensure(acc[\"dondurulmuş\"] > acc[\"sıfırdan\"] + 0.03 and acc[\"ince ayar\"] > acc[\"sıfırdan\"] + 0.02, \"ilgili ön-eğitim az veride kazandırmalı\")\n        ensure(acc[\"ilgisiz kaynak\"] < acc[\"sıfırdan\"], \"ilgisiz kaynak zarar vermeli\")\n        R_.table(\"Hedef görev: 100 etiketli örnek (3 tohum)\", [\"yöntem\", \"test doğruluğu\"], [[k, f\"{v:.3f}\"] for k, v in acc.items()],\n                 note=\"Kaynak görev (gizli etkenleri tahmin etmek) hedefin ihtiyaç duyduğu gösterimi öğretir: yardımcı görevle ön-eğitim. Rastgele etiketlerle ön-eğitilen gövde ise ezberler ve kötü öznitelik üretir — aktarımın değeri kaynak ile hedefin ilişkisine bağlıdır.\")\n    R_.test(\"transfer öğrenme\", tr)\n",
 "e": "from labcheck import need, ensure, bench, fmt_time\n\n\ndef check(ns, R_):\n    def t():\n        widths = [2, 8, 32, 128]\n        b = need(ns, \"batched_cv\")(widths, 5, 150, 0)\n        l = need(ns, \"loop_cv\")(widths, 5, 150, 0)\n        ensure(set(b) == set(widths) and set(l) == set(widths), \"her genişlik için sonuç\")\n        ensure(b[128][0] > b[2][0] + 0.2 and l[128][0] > l[2][0] + 0.2, \"genişlik 2 yetersiz uyum\")\n        for w in widths[1:]:\n            ensure(abs(b[w][0] - l[w][0]) < 0.08, f\"toplu ve döngülü CV yakın (genişlik {w})\")\n        tb = bench(lambda: ns[\"batched_cv\"](widths, 5, 150, 0), repeat=1, number=1)\n        tl = bench(lambda: ns[\"loop_cv\"](widths, 5, 150, 0), repeat=1, number=1)\n        R_.table(\"digits, 5 katlı CV, tek gizli katman, 150 tam batch adım\", [\"genişlik\", \"toplu: ort. ± std\", \"döngü: ort. ± std\"], [[w, f\"{b[w][0]:.3f} ± {b[w][1]:.3f}\", f\"{l[w][0]:.3f} ± {l[w][1]:.3f}\"] for w in widths],\n                 note=f\"Süre: toplu {fmt_time(tb)}, döngü {fmt_time(tl)} (20 model). Toplu sürümde genişlikler maskeyle tek tensörde; başlatmalar farklı olduğundan sonuçlar birebir değil, istatistiksel olarak uyumlu — genişlik 2 gibi kararsız modellerde başlatma farkı büyük olabilir.\")\n    R_.test(\"toplu CV\", t)\n",
 "x1": "import numpy as np\nfrom labcheck import need, ensure, max_loops\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"topk döngüsüz\", lambda: max_loops(src, 0, func=\"topk_features\"))\n\n    def t():\n        rng = np.random.default_rng(5)\n        X = rng.normal(size=(40, 30))\n        y = (X[:, 7] > 0).astype(int)\n        ensure(7 in need(ns, \"topk_features\")(X, y, 3), \"bilgili öznitelik seçilmeli\")\n        rows, w_all, r_all = [], [], []\n        for s in range(5):\n            w, r = need(ns, \"cv_wrong_right\")(s)\n            w_all.append(w)\n            r_all.append(r)\n            rows.append([s, f\"{w:.2f}\", f\"{r:.2f}\"])\n        ensure(np.mean(w_all) > 0.85 and abs(np.mean(r_all) - 0.5) < 0.15, \"yanlış yol ~%100, doğru yol ~%50\")\n        R_.table(\"Saf gürültü (n = 50, p = 5000): 5 katlı CV doğruluğu\", [\"tohum\", \"seçim CV dışında (yanlış)\", \"seçim her katta (doğru)\"], rows,\n                 note=\"Etiketlerle ilişkili görünen 100 öznitelik şans eseri seçilir — ama doğrulama örneklerinin etiketleri de bu seçime katkı yapmıştır. Hastie, Tibshirani & Friedman (ESL §7.10.2) 'çapraz doğrulamanın yanlış ve doğru yolu'. Kural: etiket gören her adım (seçim, ölçekleme, hiperparametre) CV döngüsünün içinde.\")\n    R_.test(\"öznitelik seçimi sızıntısı\", t)\n",
 "x2": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        sizes = [15, 30, 60, 120, 500, 2000]\n        rows = []\n        res = {}\n        for d in (1, 5, 15):\n            tr, va = need(ns, \"learning_curve\")(sizes, d, 1e-8, 1)\n            res[d] = (tr, va)\n            rows.append([d] + [f\"{a:.3f}/{b:.3f}\" for a, b in zip(tr, va)])\n        ensure(need(ns, \"diagnose\")(res[1][0][-1], res[1][1][-1], 0.04) == \"yüksek yanlılık\", \"derece 1: yanlılık\")\n        ensure(ns[\"diagnose\"](res[15][0][1], res[15][1][1], 0.04) == \"yüksek varyans\", \"derece 15, az veri: varyans\")\n        ensure(res[15][1][-1] < res[15][1][1], \"varyans daha çok veriyle azalır\")\n        R_.table(\"Eğitim / doğrulama MSE (gürültü varyansı 0.04)\", [\"derece\"] + [f\"n = {n}\" for n in sizes], rows,\n                 note=\"Yanlılık: iki eğri yüksek ve birbirine yakın — daha çok veri yardım etmez, daha büyük model gerek. Varyans: büyük boşluk — veri ya da düzenleme yardım eder.\")\n    R_.test(\"öğrenme eğrileri\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        X = np.arange(64.0).reshape(1, 64)\n        A = need(ns, \"shift_augment\")(X)\n        ensure(A.shape == (5, 64) and (A[0] == X[0]).all(), \"orijinal + 4 kaydırma\")\n        I = A.reshape(5, 8, 8)\n        ensure(I[1, 1:, :].tolist() == X.reshape(8, 8)[:-1, :].tolist() and (I[1, 0] == 0).all(), \"aşağı kaydırma, sıfır dolgu\")\n        res = [(need(ns, \"few_shot_accuracy\")(False, s), ns[\"few_shot_accuracy\"](True, s)) for s in range(5)]\n        a = np.mean([r[0] for r in res])\n        b = np.mean([r[1] for r in res])\n        ensure(b > a + 0.02, \"artırma yardımcı olmalı\")\n        R_.table(\"digits, sınıf başına 3 eğitim örneği, 1-NN (5 tohum)\", [\"veri\", \"test doğruluğu\"], [[\"30 örnek\", f\"{a:.3f}\"], [\"+ ±1 piksel kaydırmalar (150)\", f\"{b:.3f}\"]],\n                 note=\"Kaydırılmış kopyalar, test rakamı biraz kaymış yazıldığında da yakın bir komşu bulunmasını sağlar: değişmezlik bilgisini veriye gömmek. (Lojistik regresyonda aynı artırma 8×8'lik küçük görüntülerde sınıfları bulandırıp zarar verebilir — artırma modelle birlikte düşünülmeli.) CNN'ler kaydırma eşdeğerliğini mimariyle (M4.1) verir.\")\n    R_.test(\"veri artırma\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m3_9_training_practice as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Kapasite ve aşırı uyum** (ders alıştırması 4): Fashion-MNIST'te 784-300-100-10, 784-400-500-300-100-10 ve 784-600-700-500-400-300-100-10 tam bağlı ağlar 40 epoch. Daha geniş/derin ağ eğitim kaybını daha hızlı sıfıra indirir; doğrulama kaybı bir noktadan sonra **yükselir**. Bu ayrışma aşırı uyumdur: model eğitim örneklerini (gürültü dahil) ezberler. Doğrulama doğruluğu çoğu zaman yakın kalır — çapraz entropi, aşırı güvenli yanlışları da cezalandırdığı için önce kayıp bozulur.

**Erken durdurma:** doğrulama kaybını izle, en iyi epoch'un ağırlıklarını sakla (kontrol noktası), `patience` epoch iyileşme yoksa dur. **Model seçimi:** eğitim/doğrulama/test; az veride **k-katlı CV** (her kat bir kez doğrulama). Test kümesi yalnız en son, bir kez.

**Transfer öğrenme:** benzer bir görevde eğitilmiş ağın alt katmanlarını yeniden kullan (alt katmanlar genel öznitelikler, üst katmanlar göreve özgü). Tarif: çıkış katmanını değiştir → yeniden kullanılan katmanları **dondur** → başlığı eğit → üstten başlayarak **çöz**, küçük öğrenme oranıyla ince ayar. Veri azsa daha çok katman dondur. Ön-eğitim çeşitleri: denetimli (ImageNet), denetimsiz/öz-denetimli (otomatik kodlayıcılar M5.6, maskeli dil modeli M4.16), yardımcı görev (kolay etiketlenen ilgili bir görevde ön-eğitim).

**Veri sızıntısı:** modelin, dağıtımda erişemeyeceği bilgiyi eğitimde görmesi. Türler: *ön işlemede* (ölçekleyici/tamamlayıcı tüm veride öğrenilir), *gruplarda* (aynı hasta/kullanıcı/görüntünün kopyaları iki tarafta), *zamanda* (gelecekten geçmişe; otokorelasyonlu seride rastgele bölme), *hedefte* (hedefin sonucu olan bir öznitelik). Belirtisi: gerçek dışı yüksek doğrulama skoru.

## 1 · Elle

<!-- çözüm -->

$\sum(a\cdot b+b)$: 235 200 + 300 + 30 000 + 100 + 1 000 + 10 = 266 610 vb. Üçüncü model 6 kat parametreyle aynı doğruluğa ulaşır ama çok daha erken aşırı uyar.

Dondurulmuş iki katmanla yalnız 100 × 10 + 10 = 1010 parametre eğitilir: binlerce kat az veriyle eğitilebilir.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Ders alıştırması 4'ün modelleri (Fashion-MNIST, 784 girdi, 10 sınıf): parametre sayısı**

- `p1`: 784-300-100-10
- `p2`: 784-400-500-300-100-10
- `p3`: 784-600-700-500-400-300-100-10

**Transfer**

- `fz`: 784-300-100-10'da iki gizli katman dondurulursa eğitilebilir parametre
- `kf`: 5 katlı CV, 3 aday model: eğitilen toplam model (son yeniden eğitim hariç)

In [ ]:
cevaplar = dict(
    p1=None,
    p2=None,
    p3=None,
    fz=None,
    kf=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"p1":[266610,0],"p2":[695910,0],"p3":[1594010,0],"fz":[1010,0],"kf":[15,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

**Grup sızıntısı**, saf Python. Her "denek" birkaç gürültülü ölçüm üretir (aynı kişinin tekrarlanan örnekleri).

- `make_groups(n_groups, per_group, seed)` → `(X, y, g)`: grup başına rastgele bir merkez ve etiket (etiket merkezle **zayıf** ilişkili), grup içi örnekler merkez + küçük gürültü.
- `nn1_predict(Xtr, ytr, Xte)` — 1-en yakın komşu.
- `split_random(n, frac, seed)` ve `split_groups(g, frac, seed)` → `(train_idx, test_idx)`.
- `accuracy_both(X, y, g, seed)` → `(rastgele bölme doğruluğu, grup bölme doğruluğu)`.

In [ ]:
%%writefile stage_b.py
import random


def make_groups(n_groups, per_group, seed):
    ...


def nn1_predict(Xtr, ytr, Xte):
    ...


def split_random(n, frac, seed):
    ...


def split_groups(g, frac, seed):
    ...


def accuracy_both(X, y, g, seed):
    ...


X, y, g = make_groups(40, 5, 0)
print(accuracy_both(X, y, g, 0))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy.

- `kfold_indices(n, k, seed)` → k `(train, val)` çifti (karıştırılmış, katlar eşit ±1).
- `poly_cv(x, y, degrees, k=5, seed=0)` → `{derece: ortalama doğrulama MSE}` — polinom en küçük kareleri (`np.vander` + `lstsq`); döngü katlar ve dereceler üzerinde.
- `early_stop(val_losses, patience)` → `(durma_epoch'u, en_iyi_epoch)` — en iyi sonrası `patience` epoch iyileşme yoksa durulan epoch.
- `standardize_leak(Xtr, Xte)` ve `standardize_ok(Xtr, Xte)` — ilki istatistikleri birleşik veriden, ikincisi yalnız eğitimden.

In [ ]:
%%writefile stage_c.py
import numpy as np


def kfold_indices(n, k, seed):
    ...


def poly_cv(x, y, degrees, k=5, seed=0):
    ...


def early_stop(val_losses, patience):
    ...


def standardize_leak(Xtr, Xte):
    ...


def standardize_ok(Xtr, Xte):
    ...


print(early_stop([1.0, 0.8, 0.7, 0.72, 0.75, 0.71, 0.9], 2))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch, sklearn `load_digits` (8×8 = 64 öznitelik; Fashion-MNIST'in küçük vekili).

- `capacity_runs(epochs=40, seed=0)` → `{mimari: [(eğitim kaybı, doğrulama kaybı, doğrulama doğruluğu) epoch başına]}` — mimariler ders alıştırmasının ölçeklenmiş hâli: 64-300-100-10, 64-400-500-300-100-10, 64-600-700-500-400-300-100-10; Adam 1e-3, batch 32, eğitim 1000 / doğrulama 797.
- `transfer(n_target=100, seed=0)` → `{"sıfırdan", "dondurulmuş", "ince ayar", "ilgisiz kaynak"}` doğrulukları. **Yardımcı görevle ön-eğitim:** gizli 4 boyutlu etkenler $z$, gözlem $x=\sin(2\tanh(2zA_1)A_2)+\text{gürültü}\in\mathbb R^{64}$ (sabit rastgele $A_1, A_2$). Kaynak (bol etiket, 5000): $z$'yi regresyonla tahmin et (gövde 64-128-32). Hedef (az etiket, `n_target`): $\mathbb 1[z_3+z_4>0]$. (a) sıfırdan, (b) gövde dondurulmuş + yeni başlık, (c) önce başlık sonra tümü 1e−4 ile, (d) gövde **rastgele etiketlerle** ön-eğitilmiş (dondurulmuş).

Not: digits'te 0–4'ten 5–9'a aktarım burada **zarar** verir (piksel uzayı zaten neredeyse doğrusal ayrılabilir, kaynak öznitelikleri 5 sınıfa özelleşir) — negatif aktarım da bir derstir.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn as nn
from sklearn.datasets import load_digits


def capacity_runs(epochs=40, seed=0):
    ...


def transfer(n_target=100, seed=0):
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

**Toplu çapraz doğrulama:** k kat × c kapasite modeli, `torch.bmm` ile tek optimizasyonda.

- `batched_cv(widths, k=5, epochs=60, seed=0)` → `{genişlik: (ortalama doğrulama doğruluğu, std)}` — tek gizli katmanlı MLP'ler (genişlik farklı → en büyük genişlikte tensör, fazlası maskeli), her kat kendi eğitim/doğrulama indeksleri.
- `loop_cv(widths, k=5, epochs=60, seed=0)` — aynısı döngüyle.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn as nn
from sklearn.datasets import load_digits


def batched_cv(widths, k=5, epochs=60, seed=0):
    ...


def loop_cv(widths, k=5, epochs=60, seed=0):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Çapraz doğrulamanın yanlış yolu: öznitelik seçimi sızıntısı

**Saf gürültü verisi:** n = 50 örnek, p = 5000 rastgele öznitelik, rastgele ikili etiket — gerçek doğruluk %50.

- `topk_features(X, y, k)` — etiketle mutlak kovaryansı en büyük k öznitelik (döngüsüz).
- `nearest_centroid_acc(Xtr, ytr, Xte, yte)`.
- `cv_wrong_right(seed, n=50, p=5000, k=100, folds=5)` → `(yanlış yol doğruluğu, doğru yol doğruluğu)`: yanlış yol öznitelikleri **tüm** veriyle bir kez seçip sonra CV yapar; doğru yol seçimi her katın eğitim kısmında yapar.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def topk_features(X, y, k):
    ...


def nearest_centroid_acc(Xtr, ytr, Xte, yte):
    ...


def cv_wrong_right(seed, n=50, p=5000, k=100, folds=5):
    ...


print(cv_wrong_right(0))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Öğrenme eğrileri: daha çok veri mi, daha büyük model mi?

Gürültülü $\sin$ verisi; ridge polinom regresyonu.

- `learning_curve(sizes, degree, lam=1e-6, seed=0)` → `(eğitim MSE'leri, doğrulama MSE'leri)`.
- `diagnose(train_err, val_err, noise_var)` → `"yüksek yanlılık"` (ikisi de gürültüden çok büyük ve yakın) ya da `"yüksek varyans"` (büyük boşluk).

In [ ]:
%%writefile extra_x2.py
import numpy as np


def learning_curve(sizes, degree, lam=1e-6, seed=0):
    ...


def diagnose(train_err, val_err, noise_var):
    ...


print(learning_curve([20, 100], 3))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Az veriyle veri artırma (1-NN)

sklearn `load_digits`; sınıf başına **3** eğitim örneği, 1-en yakın komşu.

- `shift_augment(X)` — 8×8 görüntüleri ±1 piksel 4 yöne kaydır (sıfır dolgu) → 5 kat veri.
- `few_shot_accuracy(augment, seed, n_per_class=3)` — 1-NN (`sklearn.neighbors.KNeighborsClassifier(1)`), test doğruluğu.

In [ ]:
%%writefile extra_x3.py
import numpy as np
from sklearn.datasets import load_digits
from sklearn.neighbors import KNeighborsClassifier


def shift_augment(X):
    ...


def few_shot_accuracy(augment, seed, n_per_class=3):
    ...


print(few_shot_accuracy(False, 0), few_shot_accuracy(True, 0))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Yosinski ve ark. (2014): Özellikler ne kadar aktarılabilir?

[Yosinski, Clune, Bengio, Lipson — How transferable are features in deep neural networks? (NeurIPS 2014)](https://arxiv.org/abs/1411.1792)

Makalenin A/B bölünmesi deneyini küçük ölçekte yeniden üret: CIFAR-100'ü (ya da Fashion-MNIST'i) iki rastgele sınıf yarısına böl, iki taban ağ eğit (baseA, baseB), sonra n = 1…L katmanı kopyalayıp (i) dondurarak (AnB) ve (ii) ince ayarla (AnB+) B görevinde eğit; ayrıca BnB ve BnB+ kontrollerini yap. Doğruluk – n eğrisini makalenin Şekil 2'siyle nitel karşılaştır: "kırılgan birlikte uyum" (fragile co-adaptation) ve "özgüllük" etkilerini ayır.


**Adımlar**

1. d aşamasındaki transfer fonksiyonunu katman sayısı parametresiyle genelleştir.
2. Her nokta için en az 3 tohum.

**Kabul ölçütleri**

- BnB orta katmanlarda düşer (birlikte uyum), BnB+ toparlanır.
- AnB+ taban ağdan iyi ya da eşit (transferin genelleme bonusu).

## Makale ödevi: Kaufman ve ark. (2012): Veri madenciliğinde sızıntı

[Kaufman, Rosset, Perlich — Leakage in Data Mining (TKDD 2012)](https://doi.org/10.1145/2382577.2382579)

Makaledeki sızıntı sınıflandırmasını (öznitelik sızıntısı, eğitim örneği sızıntısı) kullanarak bu modüldeki üç örneği (b, x1 ve ölçekleme) sınıfla; gerçek bir Kaggle yarışması sızıntısını (makalede örnekler var) özetleyip kendi küçük veri kümende yeniden canlandır. "Öğrenme-tahmin ayrımı" (learn-predict separation) ilkesini bir scikit-learn `Pipeline` + `GroupKFold`/`TimeSeriesSplit` ile uygula.


**Adımlar**

1. Sızıntılı ve sızıntısız skorları aynı tabloda raporla.

**Kabul ölçütleri**

- Her sızıntı türü için skor farkı ve düzeltilmiş protokol.

## Kaynaklar

- [Géron — Hands-On Machine Learning (3. baskı), Bölüm 10–11](https://github.com/ageron/handson-ml3) — Apache-2.0 (kod)
- [Yosinski, Clune, Bengio, Lipson — How transferable are features in deep neural networks? (NeurIPS 2014)](https://arxiv.org/abs/1411.1792) — arXiv
- [Kaufman, Rosset, Perlich — Leakage in Data Mining: Formulation, Detection, and Avoidance (KDD 2011 / TKDD 2012)](https://doi.org/10.1145/2382577.2382579) — ©
- [Prechelt — Early Stopping — But When? (Neural Networks: Tricks of the Trade 1998)](https://doi.org/10.1007/3-540-49430-8_3) — ©
- [scikit-learn — Cross-validation: evaluating estimator performance](https://scikit-learn.org/stable/modules/cross_validation.html) — BSD-3